# Build a Custom OpenAI Chatbot with ML-Driven Prompt Engineering

The code below is designed to run as-is with one exception: **the OpenAI API key must be specified**. Edit the cell below to add your API key between the double quotes.

Then, to execute each code cell, click on it and press `Shift` + `Enter` on your keyboard.

In [121]:
import openai
openai.api_base = "https://api.openai.com/v1"
openai.api_key = ""

## Step 0: Inspecting Non-Customized Results

Before we perform any prompt engineering, **let's ask the OpenAI model some questions and see how it answers**.

(If you encounter an `AuthenticationError` when running this code, make sure that you have added a valid API key to the cell above and executed it.)

In [122]:
ukraine_prompt = """
Question: "When did Russia invade Ukraine?"
Answer:
"""
initial_ukraine_answer = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=ukraine_prompt,
    max_tokens=150
)["choices"][0]["text"].strip()
print(initial_ukraine_answer)

Russia invaded Ukraine in February 2014, starting with the occupation of Crimea and later expanding into eastern Ukraine.


In [123]:
twitter_prompt = """
Question: "Who owns Twitter?"
Answer:
"""
initial_twitter_answer = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=twitter_prompt,
    max_tokens=150
)["choices"][0]["text"].strip()
print(initial_twitter_answer)

As of 2021, the CEO and co-founder of Twitter, Jack Dorsey, owns a 2.3% stake in the company, making him the largest individual shareholder. Other major shareholders include investment firms such as Vanguard, Morgan Stanley, and BlackRock. However, since Twitter is a publicly traded company, its ownership is spread among millions of shareholders.


The model is answering this way because the training data ends in 2021. **Our task will be to provide context from 2022 to help the model answer these questions correctly.**

# Step 1: Prepare Dataset

## Loading and Wrangling Data

**The data should be loaded into a pandas `DataFrame` called `df` where each row represents a text sample, and there is only one column, `"text"`, which contains the raw text data.**

In this particular case we are collecting data from [the Wikipedia page for the year 2022](https://en.wikipedia.org/wiki/2022) and performing some data wrangling to get it into the appropriate format. Don't worry too much about the details here, since data wrangling looks different for every dataset!

In [124]:
from dateutil.parser import parse
import pandas as pd
import requests

# Get the Wikipedia page for "2022" since OpenAI's models stop in 2021
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
resp = requests.get("https://en.wikipedia.org/w/api.php?action=query&prop=extracts&exlimit=1&titles=2022&explaintext=1&formatversion=2&format=json", headers=headers)

# Debug: Check response status and content
print(f"Status Code: {resp.status_code}")
print(f"Response Headers: {resp.headers}")
print(f"Response Text (first 500 chars): {resp.text[:500]}")

resp.json()["query"]["pages"][0]["extract"]

# Load page text into a dataframe
df = pd.DataFrame()
df["text"] = resp.json()["query"]["pages"][0]["extract"].split("\n")

# Clean up text to remove empty lines and headings
df = df[(df["text"].str.len() > 0) & (~df["text"].str.startswith("=="))]

# In some cases dates are used as headings instead of being part of the
# text sample; adjust so dated text samples start with dates
prefix = ""
for (i, row) in df.iterrows():
    # If the row already has " - ", it already has the needed date prefix
    if " – " not in row["text"]:
        try:
            # If the row's text is a date, set it as the new prefix
            parse(row["text"])
            prefix = row["text"]
        except:
            # If the row's text isn't a date, add the prefix
            row["text"] = prefix + " – " + row["text"]
df = df[df["text"].str.contains(" – ")]
df
df.to_csv("2022_wikipedia_data.csv", index=False)

Status Code: 200
Response Headers: {'date': 'Sun, 22 Feb 2026 03:01:07 GMT', 'server': 'mw-api-ext.codfw.main-76466597fc-dsn7j', 'x-content-type-options': 'nosniff', 'content-security-policy': "default-src 'self'; script-src 'none'; object-src 'none'", 'x-frame-options': 'DENY', 'content-disposition': 'inline; filename=api-result.json', 'cache-control': 'private, must-revalidate, max-age=0', 'content-type': 'application/json; charset=utf-8', 'content-encoding': 'gzip', 'age': '2', 'x-cache': 'cp4044 miss, cp4044 pass', 'x-cache-status': 'pass', 'server-timing': 'cache;desc="pass", host;desc="cp4044"', 'strict-transport-security': 'max-age=106384710; includeSubDomains; preload', 'report-to': '{ "group": "wm_nel", "max_age": 604800, "endpoints": [{ "url": "https://intake-logging.wikimedia.org/v1/events?stream=w3c.reportingapi.network_error&schema_uri=/w3c/reportingapi/network_error/1.0.0" }] }', 'nel': '{ "report_to": "wm_nel", "max_age": 604800, "failure_fraction": 0.05, "success_fracti

## Generating Embeddings

We'll use the `Embedding` tooling from OpenAI [documentation here](https://platform.openai.com/docs/guides/embeddings/embeddings) to create vectors representing each row of our custom dataset.

In order to avoid a `RateLimitError` we'll send our data in batches to the `Embedding.create` function.

In [125]:
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"
batch_size = 100
embeddings = []
for i in range(0, len(df), batch_size):
    # Send text data to OpenAI model to get embeddings
    response = openai.Embedding.create(
        input=df.iloc[i:i+batch_size]["text"].tolist(),
        engine=EMBEDDING_MODEL_NAME
    )
    
    # Add embeddings to list
    embeddings.extend([data["embedding"] for data in response["data"]])

# Add embeddings list to dataframe
df["embeddings"] = embeddings
df

,text,embeddings
0,– 2022 (MMXXII) was a common year starting on...,"[5.03144838148728e-05, -0.017939811572432518, ..."
1,– The year began with another wave in the COV...,"[-0.004297760780900717, -0.01981227844953537, ..."
2,– 2022 was also dominated by wars and armed c...,"[-0.008323960937559605, -0.015191062353551388,..."
7,– The Russo-Ukrainian war escalated after Rus...,"[-0.015421273186802864, -0.004974063951522112,..."
15,January 1 – France takes over the Presidency ...,"[0.030140550807118416, -0.010628909803926945, ..."
...,...,...
270,December 21–December 26 – A major winter storm...,"[-0.024808574467897415, -0.023849913850426674,..."
271,December 24 – 2022 Fijian general election: Th...,"[-0.01166312675923109, -0.00934850424528122, -..."
272,December 31 – Former Pope Benedict XVI dies at...,"[0.02359509840607643, 0.007731214631348848, -0..."
276,December 7 – The world population was estimate...,"[-0.004124105907976627, -0.014428064227104187,..."


In order to avoid having to run that code again in the future, we'll save the generated embeddings as a CSV file.

In [126]:
df.to_csv("embeddings-case_study.csv")

In [127]:
! ls

'ls' is not recognized as an internal or external command,
operable program or batch file.


If you want to stop the tutorial here and come back, you can reload `df` using this code (again adding your API key) rather than generating the embeddings again:

In [128]:
# import numpy as np
# import pandas as pd
# import openai
# openai.api_base = "https://openai.vocareum.com/v1"
# openai.api_key = "YOUR API KEY"
# df = pd.read_csv("embeddings-case_study.csv", index_col=0)
# df["embeddings"] = df["embeddings"].apply(eval).apply(np.array)

In [129]:
df

,text,embeddings
0,– 2022 (MMXXII) was a common year starting on...,"[5.03144838148728e-05, -0.017939811572432518, ..."
1,– The year began with another wave in the COV...,"[-0.004297760780900717, -0.01981227844953537, ..."
2,– 2022 was also dominated by wars and armed c...,"[-0.008323960937559605, -0.015191062353551388,..."
7,– The Russo-Ukrainian war escalated after Rus...,"[-0.015421273186802864, -0.004974063951522112,..."
15,January 1 – France takes over the Presidency ...,"[0.030140550807118416, -0.010628909803926945, ..."
...,...,...
270,December 21–December 26 – A major winter storm...,"[-0.024808574467897415, -0.023849913850426674,..."
271,December 24 – 2022 Fijian general election: Th...,"[-0.01166312675923109, -0.00934850424528122, -..."
272,December 31 – Former Pope Benedict XVI dies at...,"[0.02359509840607643, 0.007731214631348848, -0..."
276,December 7 – The world population was estimate...,"[-0.004124105907976627, -0.014428064227104187,..."


# Step 2: Create a Function that Finds Related Pieces of Text for a Given Question

What we are implementing here is similar to a search engine or recommendation algorithm. We want to sort all of the rows of our dataset from least relevant to most relevant.

This will use the embeddings that we generated previously in order to compare the vectorized version of our question to the vectorized versions of the rows of the dataset.

In [130]:
from openai.embeddings_utils import get_embedding, distances_from_embeddings

def get_rows_sorted_by_relevance(question, df):
    """
    Function that takes in a question string and a dataframe containing
    rows of text and associated embeddings, and returns that dataframe
    sorted from least to most relevant for that question
    """
    
    # Get embeddings for the question text
    question_embeddings = get_embedding(question, engine=EMBEDDING_MODEL_NAME)
    
    # Make a copy of the dataframe and add a "distances" column containing
    # the cosine distances between each row's embeddings and the
    # embeddings of the question
    df_copy = df.copy()
    df_copy["distances"] = distances_from_embeddings(
        question_embeddings,
        df_copy["embeddings"].values,
        distance_metric="cosine"
    )
    
    # Sort the copied dataframe by the distances and return it
    # (shorter distance = more relevant so we sort in ascending order)
    df_copy.sort_values("distances", ascending=True, inplace=True)
    return df_copy


Let's test that out for a couple different questions:

In [ ]:
get_rows_sorted_by_relevance("When did Russia invade Ukraine?", df)

In [132]:
get_rows_sorted_by_relevance("Who owns Twitter?", df)

,text,embeddings,distances
227,October 28 – Elon Musk completes his $44 billi...,"[-0.010219527408480644, -0.016063760966062546,...",0.173025
109,April 25 – Elon Musk reaches an agreement to a...,"[-0.012646312825381756, -0.01627325266599655, ...",0.180818
32,January 24 – The federal government under Scot...,"[-0.009006649255752563, -0.008588949218392372,...",0.246819
245,"November 11 – The cryptocurrency exchange FTX,...","[0.002813608618453145, -0.025335688143968582, ...",0.262609
257,"November 30 – OpenAI releases ChatGPT, an arti...","[-0.01123974658548832, -0.014409263618290424, ...",0.264680
...,...,...,...
187,August 28 – 2022 Pakistan floods: Pakistan dec...,"[-0.011937441304326057, -0.02288598194718361, ...",0.325948
194,September 5 – A 6.8 earthquake strikes Luding ...,"[0.012638013809919357, 0.016330983489751816, 0...",0.326661
198,September 12 – September 2022 Armenia–Azerbaij...,"[-0.012428279034793377, -0.009104359894990921,...",0.326854
24,January 13 – Bikaner-Guwahati Express derailme...,"[-0.0193224735558033, -0.004884106572717428, 0...",0.327207


# Step 3: Create a Function that Composes a Text Prompt

Building on that sorted list of rows, we're going to select the create a text prompt that provides context to a `Completion` model in order to help it answer a question. The outline of the prompt looks like this:

```
Answer the question based on the context below, and if the
question can't be answered based on the context, say "I don't
know"

Context:

{context}

---

Question: {question}
Answer:
```

We want to fit as much of our dataset as possible into the "context" part of the prompt without exceeding the number of tokens allowed by the `Completion` model, which is currently 4,000. So we'll loop over the dataset, counting the tokens as we go, and stop when we hit the limit. Then we'll join that list of text data into a single string and add it to the prompt.

In [133]:
import tiktoken

def create_prompt(question, df, max_token_count):
    """
    Given a question and a dataframe containing rows of text and their
    embeddings, return a text prompt to send to a Completion model
    """
    # Create a tokenizer that is designed to align with our embeddings
    tokenizer = tiktoken.get_encoding("cl100k_base")
    
    # Count the number of tokens in the prompt template and question
    prompt_template = """
Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context: 

{}

---

Question: {}
Answer:"""
    
    current_token_count = len(tokenizer.encode(prompt_template)) + \
                            len(tokenizer.encode(question))
    
    context = []
    for text in get_rows_sorted_by_relevance(question, df)["text"].values:
        
        # Increase the counter based on the number of tokens in this row
        text_token_count = len(tokenizer.encode(text))
        current_token_count += text_token_count
        
        # Add the row of text to the list if we haven't exceeded the max
        if current_token_count <= max_token_count:
            context.append(text)
        else:
            break

    return prompt_template.format("\n\n###\n\n".join(context), question)
    

Now let's test that out! We'll use a `max_token_count` below the actual limit just to keep the output shorter and more readable.

In [134]:
print(create_prompt("When did Russia invade Ukraine?", df, 200))


Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context: 

March 2 – 2022 Russian invasion of Ukraine: Russia captures its first large city, the Black Sea port of Kherson, as shelling intensifies across many parts of Ukraine, including civilian areas.

###

April 3 – 2022 Russian invasion of Ukraine: As Russia's forces retreat from areas near Kyiv, it is accused by Ukraine of war crimes, amid mounting evidence of indiscriminate civilian killings, including the Bucha massacre.

---

Question: When did Russia invade Ukraine?
Answer:


In [135]:
print(create_prompt("Who owns Twitter?", df, 100))


Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context: 

October 28 – Elon Musk completes his $44 billion acquisition of Twitter.

###

April 25 – Elon Musk reaches an agreement to acquire the social media network Twitter (which he later rebrands as X) for US$44 billion, which later closes in October.

---

Question: Who owns Twitter?
Answer:


# Step 4: Create a Function that Answers a Question

Our final step is to send that text prompt to a `Completion` model and parse the model output!

In [136]:
COMPLETION_MODEL_NAME = "gpt-3.5-turbo-instruct"

def answer_question(
    question, df, max_prompt_tokens=1800, max_answer_tokens=150
):
    """
    Given a question, a dataframe containing rows of text, and a maximum
    number of desired tokens in the prompt and response, return the
    answer to the question according to an OpenAI Completion model
    
    If the model produces an error, return an empty string
    """
    
    prompt = create_prompt(question, df, max_prompt_tokens)
    
    try:
        response = openai.Completion.create(
            model=COMPLETION_MODEL_NAME,
            prompt=prompt,
            max_tokens=max_answer_tokens
        )
        return response["choices"][0]["text"].strip()
    except Exception as e:
        print(e)
        return ""
        

Now that we have all of the code complete, let's test it out!

In [137]:
custom_ukraine_answer = answer_question("When did Russia invade Ukraine?", df)
print(custom_ukraine_answer)

2022 Russian invasion of Ukraine: Putin announces a "special military operation" to support the Russian-backed breakaway republics of Donetsk and Luhansk, whose paramilitary forces had been fighting Ukraine in the Donbas conflict since 2014.


In [138]:
custom_twitter_answer = answer_question("Who owns Twitter?", df)
print(custom_twitter_answer)

Elon Musk


Below we compare answers with and without our custom prompt:

In [139]:
print(f"""
When did Russia invade Ukraine?

Original Answer: {initial_ukraine_answer}
Custom Answer:   {custom_ukraine_answer}

Who owns Twitter?
Original Answer: {initial_twitter_answer}
Custom Answer:   {custom_twitter_answer}
""")


When did Russia invade Ukraine?

Original Answer: Russia invaded Ukraine in February 2014, starting with the occupation of Crimea and later expanding into eastern Ukraine.
Custom Answer:   2022 Russian invasion of Ukraine: Putin announces a "special military operation" to support the Russian-backed breakaway republics of Donetsk and Luhansk, whose paramilitary forces had been fighting Ukraine in the Donbas conflict since 2014.

Who owns Twitter?
Original Answer: As of 2021, the CEO and co-founder of Twitter, Jack Dorsey, owns a 2.3% stake in the company, making him the largest individual shareholder. Other major shareholders include investment firms such as Vanguard, Morgan Stanley, and BlackRock. However, since Twitter is a publicly traded company, its ownership is spread among millions of shareholders.
Custom Answer:   Elon Musk



## Summary

You just used unsupervised machine learning to perform prompt engineering for custom OpenAI chat responses!

In this example, we provided context from 2022 news headlines to answer questions about current events. Try finding your own dataset for some other custom task!

In [140]:
df

,text,embeddings
0,– 2022 (MMXXII) was a common year starting on...,"[5.03144838148728e-05, -0.017939811572432518, ..."
1,– The year began with another wave in the COV...,"[-0.004297760780900717, -0.01981227844953537, ..."
2,– 2022 was also dominated by wars and armed c...,"[-0.008323960937559605, -0.015191062353551388,..."
7,– The Russo-Ukrainian war escalated after Rus...,"[-0.015421273186802864, -0.004974063951522112,..."
15,January 1 – France takes over the Presidency ...,"[0.030140550807118416, -0.010628909803926945, ..."
...,...,...
270,December 21–December 26 – A major winter storm...,"[-0.024808574467897415, -0.023849913850426674,..."
271,December 24 – 2022 Fijian general election: Th...,"[-0.01166312675923109, -0.00934850424528122, -..."
272,December 31 – Former Pope Benedict XVI dies at...,"[0.02359509840607643, 0.007731214631348848, -0..."
276,December 7 – The world population was estimate...,"[-0.004124105907976627, -0.014428064227104187,..."
